# Run RAG on content vector db

## Imports deps

In [2]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PayloadSchemaType, PointStruct, SparseVectorParams, Document, Prefetch, FusionQuery
from qdrant_client import models

import pandas as pd
import openai
from dotenv import load_dotenv
import os

/Users/antoineestienne/GithubRepositories/ai-bootcamp-dev-repo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv()
qdrant_url = os.getenv("QDRANT_URL")
qdrant_api_key = os.getenv("QDRANT_API_KEY")
qdrant_client = QdrantClient(
    url=qdrant_url,
    api_key=qdrant_api_key
)

In [4]:
CONTENT_COLLECTION_NAME="Content-collection-00"

## Retrieve data

In [5]:
def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=[text],
        model=model,
    )
    return response.data[0].embedding

In [6]:
def retrieve_data(query, qdrant_client, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name=CONTENT_COLLECTION_NAME,
        prefetch=[
            Prefetch(
                query=query_embedding,
                using="text-embedding-3-small",
                limit=20
            ),
            Prefetch(
                query=Document(
                    text=query,
                    model="qdrant/bm25"
                ),
                using="bm25",
                limit=20
            )
        ],
        query=FusionQuery(fusion="rrf"),
        limit=k,
    )

    retrieved_context_ids = []
    retrieved_context = []  
    similarity_scores = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["id"])
        retrieved_context.append(result.payload["content_text"])
        similarity_scores.append(result.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
    }

In [7]:
results = retrieve_data("Write me a post about decentralized education", qdrant_client, k=20)
results

{'retrieved_context_ids': ['at://did:plc:a5nmb42bv7wuvjbkdlw2q3bs/app.bsky.feed.post/3m7hvn7l2es2r',
  'at://did:plc:nltab4umhiuyhp3fksrmixcz/app.bsky.feed.post/3m6nrfrmia223',
  'at://did:plc:5azjjblnblc2rivoal6uxfue/app.bsky.feed.post/3lsboxhvqim2p',
  'at://did:plc:5azjjblnblc2rivoal6uxfue/app.bsky.feed.post/3lvnve5reok22',
  'at://did:plc:a5nmb42bv7wuvjbkdlw2q3bs/app.bsky.feed.post/3m7hvn7l2es2r',
  'at://did:plc:x5bzv4h5nmmm62arpjcoxg3t/app.bsky.feed.post/3m7n5bytk7c2v',
  'at://did:plc:wxu4xn423gev5fkrs3jnk6xp/app.bsky.feed.post/3jsscfcpba22x',
  'at://did:plc:rmwtx6k7ob45dewcn35gge77/app.bsky.feed.post/3m7ov4nb5ok2p',
  'at://did:plc:xwbfvbl7ucu5r7qzwgqx2lgh/app.bsky.feed.post/3kwblehelwg2a',
  'at://did:plc:ba4ljetiaeiu5vrpfhymga7a/app.bsky.feed.post/3m7oypd3qvs24',
  'at://did:plc:xwbfvbl7ucu5r7qzwgqx2lgh/app.bsky.feed.post/3kzmdcdjpep2z',
  '1983186809183178990',
  'at://did:plc:xwbfvbl7ucu5r7qzwgqx2lgh/app.bsky.feed.post/3ljxrdra2b22t',
  'at://did:plc:odj22pn3oookfcayhqnn52

In [8]:
def process_context(context):
    formatted_context=""
    for id,chunk in zip(context["retrieved_context_ids"],context["retrieved_context"]):
        formatted_context+=f"- {id}: {chunk}\n"
    
    return formatted_context

print(process_context(results))
preprocessed_context=process_context(results)

- at://did:plc:a5nmb42bv7wuvjbkdlw2q3bs/app.bsky.feed.post/3m7hvn7l2es2r: 🌍✨ Decentralized education is the future! By leveraging zero-knowledge proofs (ZKP), we can verify skills without compromising privacy, empowering learners to showcase their credentials truly. Let’s build a transparent and equitable learning landscape together! Share your thoughts below!
- at://did:plc:nltab4umhiuyhp3fksrmixcz/app.bsky.feed.post/3m6nrfrmia223:  #Web3 #AI #Education
Andrej Karpathy: Stop trying to detect AI homework; detection is broken! Education needs to embrace AI as a co-pilot. This shift validates the need for Decentralized Learning models, where verified, on-chain credentials replace centralized, flawed gatekeepers.
- at://did:plc:5azjjblnblc2rivoal6uxfue/app.bsky.feed.post/3lsboxhvqim2p: 🌟 Unlock the power of Web3 in education! Learn about secure credentialing with blockchain and the impact of DAOs on school governance in our course. Be a pioneer in the decentralized digital age! 🌐📚

🔗 www.

## Rest of RAG piepeline

### Prompt

In [9]:
def build_prompt(preprocessed_context, question):
    prompt = f"""
You are an automated Bluesky account for The Guild.

The Guild is a peer‑run organization for software developers. We learn together, certify each other’s skills, and create opportunities through collaboration, attestations, and on‑chain credentials.

Why it matters:
- Community‑verified skills: members issue attestations that build portable, credible profiles.
- Learning by doing: contribute, earn badges, and grow through real projects.
- Open and merit‑based: progress is transparent and anchored on public infrastructure.

As a social media marketing specialist, your job is to create engaging and authentic posts for The Guild's Bluesky feed. 
Generate creative and informative social posts that highlight Guild values, activities, and member achievements, and help grow interest and participation.

Instructions:
- Base your post only on the provided information about activities, topics, and resources.
- Refer to “The Guild”, “Guild members”, projects, attestations, or accomplishments as appropriate for a real community post.
- The post should be captivating, social, and concise (target 220 characters including spaces, tags, and links to stay within Bluesky’s 300-character cap).
- Use an insightful or positive tone and encourage engagement or participation. 
- Do NOT sound like a generic bot.

Guild Activities/Topics:
{preprocessed_context}

Prompt:
{question}
"""
    return prompt

### Answer

In [10]:
def generate_answer(prompt):

    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=1.0  # High temperature for more creative output
    )
    return response.choices[0].message.content

### RAG Pipeline

In [11]:
def rag_pipeline(question,top_k=5):
    retrieved_context=retrieve_data(question,qdrant_client,top_k)
    preprocessed_context=process_context(retrieved_context)
    print(preprocessed_context)
    prompt=build_prompt(preprocessed_context,question)
    answer=generate_answer(prompt)
    return answer

In [12]:
print(rag_pipeline("what would be a good post for the upcoming end of year holidays?",top_k=10))

- at://did:plc:mxdqvruhze7qxiessj6isga2/app.bsky.feed.post/3m73yhzmjsc2m: Happy holidays!
- at://did:plc:psr5g4fnieaexafuhhkeqyrn/app.bsky.feed.post/3lnf2jxuuos2p: Great post, this could be applied to Ethereum, self sovereign ideals, markets and social goods, the analysis in the second part highlights the issues of the current conflicts in direction of markets and social needs, which end up eventually being completely different.
- at://did:plc:ujzy6umdid424ijcxfla4zu6/app.bsky.feed.post/3m63mdifzxs2i: I am so down for this -- I've been releasing software for 20 years under OSI-approved licenses, with the hope that this will empower individual users; but those who end up benefiting the most are likely Big Corps. We need a post open-source movement and better licenses.
- at://did:plc:vsgr3rwyckhiavgqzdcuzm6i/app.bsky.feed.post/3m6ynnth7vs25: At least I don't use any Meta products. Slipped that net.
- at://did:plc:xs7a7xrw46ueqfrgsnwbgwud/app.bsky.feed.post/3m7anb77ygx2w: 2025 was… a year